# Chapter 04 - pandas로 데이터에 질문하기

## 질문
**어떤 상품 카테고리의 completed 주문 기준 금액이 가장 큰가?**

- CSV: customers, products, orders, order_items
- 금액: `quantity × unit_price`
- 범위: `order_status == "completed"`
- 집계: 카테고리 / 상품 / 월 / 고객

> Evidence 1: `images/step01_question.png`


## STEP 1~2. 환경, 데이터, 실제 컬럼 확인


In [ ]:
import sys
from pathlib import Path
import pandas as pd

current = Path.cwd().resolve()
project_root = current
while project_root != project_root.parent and not (project_root / "data").is_dir():
    project_root = project_root.parent
if not (project_root / "data").is_dir():
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다.")

DATA_DIR = project_root / "data" / "raw"
REPORT_DIR = project_root / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")

datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}

print("Python:", sys.executable)
print("프로젝트 루트:", project_root)
for name, df in datasets.items():
    print("\n", name, df.shape)
    print(df.columns.tolist())

print("\n[city]")
print(customers["city"].value_counts(dropna=False))
print("\n[order_status]")
print(orders["order_status"].value_counts(dropna=False))


### 확인
실제 컬럼명과 필터 값을 먼저 확인했다. 이후 코드는 이 출력 결과를 기준으로 작성한다.


## STEP 3~4. 필터링·정렬·`line_total`


In [ ]:
customer_basic = customers[["customer_id", "gender", "age", "city"]].copy()
customers_over_30 = customers[customers["age"] >= 30].copy()
city_customers = customers[customers["city"].isin(["서울", "부산"])].copy()
top10_products = products.sort_values("price", ascending=False).head(10)

order_items = order_items.copy()
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

print("30세 이상:", len(customers_over_30))
print("서울/부산:", len(city_customers))
display(top10_products)
display(order_items[["order_id","product_id","quantity","unit_price","line_total"]].head(10))
print("전체 주문 상세 금액:", order_items["line_total"].sum())


### 해석
현재 `line_total` 합계에는 주문 상태가 반영되지 않았다. 따라서 아직 completed 주문 금액이라고 부르면 안 된다.

> Evidence 2: `images/step02_transform.png`


## STEP 5. orders merge 검증


In [ ]:
print("orders.order_id 중복:", orders["order_id"].duplicated().sum())

order_sales = order_items.merge(
    orders[["order_id","customer_id","order_date","order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("병합 전:", len(order_items))
print("병합 후:", len(order_sales))
print(order_sales["_merge"].value_counts(dropna=False))

assert len(order_items) == len(order_sales)
assert (order_sales["_merge"] == "both").all()

order_sales = order_sales.drop(columns="_merge")


### 해석
오른쪽 `orders.order_id`가 고유한지 확인하고 `many_to_one`으로 병합했다. 병합 전후 행 수와 미매칭도 확인했다.

> Evidence 3: `images/step03_merge.png`


## STEP 6~7. completed 필터 + products merge


In [ ]:
order_sales["order_date"] = pd.to_datetime(order_sales["order_date"], errors="coerce")
print("날짜 변환 실패:", order_sales["order_date"].isna().sum())

completed_order_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()

completed_order_sales["order_month"] = (
    completed_order_sales["order_date"].dt.to_period("M").astype(str)
)

print(completed_order_sales["order_status"].value_counts(dropna=False))
print("products.product_id 중복:", products["product_id"].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("products 병합 전:", len(completed_order_sales))
print("products 병합 후:", len(completed_sales_items))
print(completed_sales_items["_merge"].value_counts(dropna=False))

assert len(completed_order_sales) == len(completed_sales_items)
assert (completed_sales_items["_merge"] == "both").all()

completed_sales_items = completed_sales_items.drop(columns="_merge")


### 해석
이후 `total_sales`는 completed 주문 상세의 `quantity × unit_price` 합계다. 회계상 순매출이나 실제 회사 매출이라고 단정하지 않는다.


## STEP 8~10. 카테고리·상품·월·고객 집계


In [ ]:
category_sales = (
    completed_sales_items
    .groupby("category", as_index=False)
    .agg(total_quantity=("quantity","sum"), total_sales=("line_total","sum"))
    .sort_values("total_sales", ascending=False)
)
category_sales["sales_ratio"] = (
    category_sales["total_sales"] / category_sales["total_sales"].sum() * 100
).round(2)

product_sales = (
    completed_sales_items
    .groupby(["product_id","product_name","category"], as_index=False)
    .agg(total_quantity=("quantity","sum"), total_sales=("line_total","sum"))
    .sort_values("total_sales", ascending=False)
)

monthly_summary = (
    completed_order_sales
    .groupby("order_month", as_index=False)
    .agg(total_sales=("line_total","sum"), order_count=("order_id","nunique"))
    .sort_values("order_month")
)
monthly_summary["average_order_value"] = (
    monthly_summary["total_sales"] / monthly_summary["order_count"]
).round(0)

customer_sales = (
    completed_order_sales
    .groupby("customer_id", as_index=False)
    .agg(order_count=("order_id","nunique"), total_sales=("line_total","sum"))
    .sort_values("total_sales", ascending=False)
)
customer_sales = customer_sales.merge(
    customers[["customer_id","city"]],
    on="customer_id",
    how="left",
    validate="one_to_one",
)
customer_sales["customer_label"] = "Customer " + customer_sales["customer_id"].astype(str)

display(category_sales)
display(product_sales.head(10))
display(monthly_summary)
display(customer_sales[["customer_label","city","order_count","total_sales"]].head(10))

print("날짜 범위:", completed_order_sales["order_date"].min(),
      "~", completed_order_sales["order_date"].max())


### 해석
`total_sales`가 크다고 가장 인기 있거나 가장 수익성이 높다고 바로 단정할 수 없다. 판매 수량, 단가, 기간, 원가 등 추가 확인이 필요하다.

> Evidence 4: `images/step04_groupby.png`


## STEP 11. 총합 교차 검증


In [ ]:
base_total = completed_order_sales["line_total"].sum()
checks = pd.Series({
    "base": base_total,
    "category": category_sales["total_sales"].sum(),
    "product": product_sales["total_sales"].sum(),
    "month": monthly_summary["total_sales"].sum(),
    "customer": customer_sales["total_sales"].sum(),
}, name="total_sales")

display(checks.to_frame())
print("총합 모두 일치:", checks.nunique() == 1)
assert checks.nunique() == 1


### 해석
같은 completed 범위의 여러 집계 총합이 원본과 일치하는지 확인했다. 이는 현재 확인 가능한 merge/groupby 중복·누락 가능성을 줄이는 검증이다.

> Evidence 5: `images/step05_total_check.png`


## STEP 12. 결과 CSV 저장 및 재확인


In [ ]:
files = {
    "category": REPORT_DIR / "ch04_category_sales.csv",
    "product": REPORT_DIR / "ch04_product_sales.csv",
    "monthly": REPORT_DIR / "ch04_monthly_sales.csv",
    "customer": REPORT_DIR / "ch04_customer_sales.csv",
}

category_sales.to_csv(files["category"], index=False, encoding="utf-8-sig")
product_sales.to_csv(files["product"], index=False, encoding="utf-8-sig")
monthly_summary.to_csv(files["monthly"], index=False, encoding="utf-8-sig")
customer_sales.to_csv(files["customer"], index=False, encoding="utf-8-sig")

for name, path in files.items():
    print(name, path.exists(), path.stat().st_size)

saved = pd.read_csv(files["category"])
print(saved.shape)
print(saved.columns.tolist())
display(saved.head())


## STEP 13. LLM 코드 검증


In [ ]:
# ChatGPT 제안 코드: completed 주문 기준 월별 금액
llm_items = order_items[["order_id","quantity","unit_price"]].copy()
llm_items["line_total"] = llm_items["quantity"] * llm_items["unit_price"]

llm_sales = llm_items.merge(
    orders[["order_id","order_date","order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("병합 전/후:", len(llm_items), len(llm_sales))
print(llm_sales["_merge"].value_counts(dropna=False))

llm_sales["order_date"] = pd.to_datetime(llm_sales["order_date"], errors="coerce")
print("날짜 변환 실패:", llm_sales["order_date"].isna().sum())

llm_completed = llm_sales[llm_sales["order_status"] == "completed"].copy()
llm_completed["order_month"] = llm_completed["order_date"].dt.to_period("M").astype(str)

llm_monthly = (
    llm_completed
    .groupby("order_month", as_index=False)
    .agg(total_sales=("line_total","sum"), order_count=("order_id","nunique"))
    .sort_values("order_month")
)

compare = monthly_summary[["order_month","total_sales","order_count"]].merge(
    llm_monthly,
    on="order_month",
    how="outer",
    suffixes=("_mine","_llm"),
    indicator=True,
)
compare["sales_diff"] = compare["total_sales_mine"] - compare["total_sales_llm"]
compare["order_count_diff"] = compare["order_count_mine"] - compare["order_count_llm"]

display(compare)
print(
    "내 결과와 LLM 결과 일치:",
    (compare["_merge"] == "both").all()
    and (compare["sales_diff"] == 0).all()
    and (compare["order_count_diff"] == 0).all()
)


### LLM 검토 결과
LLM 코드가 실행됐다는 사실만으로 정답이라고 보지 않았다. 실제 컬럼명, merge key, `many_to_one`, completed 범위, `order_id.nunique()`, 총합을 직접 비교했다.

최종 판단: **검증 후 사용**

> Evidence 6: `images/step06_llm_validation.png`


## 최종 인사이트용 수치


In [ ]:
top_category = category_sales.iloc[0]
top_product = product_sales.iloc[0]
top_month = monthly_summary.loc[monthly_summary["total_sales"].idxmax()]

print("상위 카테고리:", top_category.to_dict())
print("상위 상품:", top_product.to_dict())
print("금액이 가장 큰 월:", top_month.to_dict())


## 최종 인사이트

1. 카테고리별 completed 주문 기준 금액에는 차이가 있다. 위 `category_sales`와 최종 수치 셀에서 가장 큰 카테고리를 확인한다.
2. 월별 금액은 주문 건수와 평균 주문 금액을 함께 봐야 한다. 단순히 월별 `total_sales`만 보고 증가·감소를 단정하지 않는다.

### 추가로 확인하고 싶은 질문
상위 카테고리의 금액이 큰 이유가 판매 수량 때문인지, 상품 단가 때문인지 더 확인하고 싶다.

### 현재 결과의 한계
현재 `total_sales`는 completed 주문의 `quantity × unit_price` 합계다. 할인, 배송비, 세금, 부분 환불, 원가와 마진을 반영한 회계상 순매출이나 이익은 아니다.
